# Submission Ensemble Variants

CatBoost alone scored slightly below the boosted balanced submission. This notebook creates small label-level ensembles from existing submissions, with boosted balanced as the anchor. These variants are cheap to test and do not re-train models.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ID_COL = 'id'
TARGET_COL = 'health_condition'

paths = {
    'boosted': Path('data/submission_boosted_balanced.csv'),
    'catboost': Path('data/submission_catboost_balanced_fast.csv'),
    'nn': Path('data/submission_neural_network.csv'),
    'tuned_fit_up': Path('data/submission_tuned_balanced_fit_up.csv'),
    'tuned_unhealthy_up': Path('data/submission_tuned_balanced_unhealthy_up.csv'),
}

subs = {name: pd.read_csv(path) for name, path in paths.items()}
base = subs['boosted'].copy()

for name, df in subs.items():
    assert list(df[ID_COL]) == list(base[ID_COL]), f'id order mismatch: {name}'

summary = []
for name, df in subs.items():
    dist = df[TARGET_COL].value_counts(normalize=True).mul(100).round(2).to_dict()
    summary.append({'submission': name, **dist})
pd.DataFrame(summary).fillna(0)


,submission,at-risk,unhealthy,fit
0,boosted,75.61,13.86,10.53
1,catboost,75.59,14.02,10.39
2,nn,73.12,14.96,11.92
3,tuned_fit_up,74.99,13.84,11.17
4,tuned_unhealthy_up,74.89,14.59,10.52


## Helper Functions

In [2]:
def save_variant(name, labels):
    out = base.copy()
    out[TARGET_COL] = labels
    path = Path(f'data/submission_ensemble_{name}.csv')
    out.to_csv(path, index=False)
    changed = (out[TARGET_COL] != base[TARGET_COL]).sum()
    dist = out[TARGET_COL].value_counts(normalize=True).mul(100).round(2).to_dict()
    return {
        'variant': name,
        'path': str(path),
        'changed_vs_boosted': int(changed),
        'changed_pct': round(changed / len(out) * 100, 2),
        'at-risk': dist.get('at-risk', 0),
        'fit': dist.get('fit', 0),
        'unhealthy': dist.get('unhealthy', 0),
    }

def majority_three_with_boosted_fallback(boosted, other_a, other_b):
    # For three labels where boosted is the fallback, the only time boosted changes
    # is when the two non-boosted sources agree with each other.
    return pd.Series(np.where(other_a.values == other_b.values, other_a.values, boosted.values), index=boosted.index)


## Create Ensembles

The best public score so far is boosted balanced, so every variant keeps it as the fallback when signals conflict.


In [3]:
b = subs['boosted'][TARGET_COL]
c = subs['catboost'][TARGET_COL]
n = subs['nn'][TARGET_COL]
fit_up = subs['tuned_fit_up'][TARGET_COL]
unhealthy_up = subs['tuned_unhealthy_up'][TARGET_COL]

reports = []

# 1. Three-model majority vote; ties fall back to boosted.
vote_3 = majority_three_with_boosted_fallback(b, c, n)
reports.append(save_variant('majority_boosted_catboost_nn', vote_3))

# 2. Very conservative: change boosted only when CatBoost and NN agree against boosted.
cat_nn_agree = (c == n) & (c != b)
labels = b.copy()
labels.loc[cat_nn_agree] = c.loc[cat_nn_agree]
reports.append(save_variant('catboost_nn_agree_only', labels))

# 3. Conservative minority rescue: only switch boosted at-risk rows to a minority class if CatBoost and NN agree.
minority_rescue = (b == 'at-risk') & (c == n) & (c != 'at-risk')
labels = b.copy()
labels.loc[minority_rescue] = c.loc[minority_rescue]
reports.append(save_variant('minority_rescue_catboost_nn', labels))

# 4. Fit consensus: boosted + fit_up + CatBoost majority, useful if public wants slightly more fit.
vote_fit = majority_three_with_boosted_fallback(b, fit_up, c)
reports.append(save_variant('boosted_fitup_catboost_vote', vote_fit))

# 5. Unhealthy consensus: boosted + unhealthy_up + CatBoost majority, useful if public wants slightly more unhealthy.
vote_unhealthy = majority_three_with_boosted_fallback(b, unhealthy_up, c)
reports.append(save_variant('boosted_unhealthyup_catboost_vote', vote_unhealthy))

report = pd.DataFrame(reports)
report


,variant,path,changed_vs_boosted,changed_pct,at-risk,fit,unhealthy
0,majority_boosted_catboost_nn,data/submission_ensemble_majority_boosted_catb...,3955,1.34,75.21,10.76,14.03
1,catboost_nn_agree_only,data/submission_ensemble_catboost_nn_agree_onl...,3955,1.34,75.21,10.76,14.03
2,minority_rescue_catboost_nn,data/submission_ensemble_minority_rescue_catbo...,2559,0.87,74.75,10.97,14.28
3,boosted_fitup_catboost_vote,data/submission_ensemble_boosted_fitup_catboos...,1552,0.52,75.37,10.76,13.87
4,boosted_unhealthyup_catboost_vote,data/submission_ensemble_boosted_unhealthyup_c...,1690,0.57,75.30,10.52,14.18


## Disagreement Diagnostics

In [4]:
diagnostics = pd.DataFrame({
    'pair': ['boosted vs catboost', 'boosted vs nn', 'catboost vs nn'],
    'disagreement_pct': [
        round((b != c).mean() * 100, 2),
        round((b != n).mean() * 100, 2),
        round((c != n).mean() * 100, 2),
    ]
})
diagnostics


,pair,disagreement_pct
0,boosted vs catboost,2.24
1,boosted vs nn,5.62
2,catboost vs nn,5.18
